In [170]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from scipy.sparse import hstack
from spellchecker import SpellChecker
import string
import re
import spacy


In [171]:
# Load your dataset
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
#filling missing values
train.fillna(' ', inplace=True)
test.fillna(' ', inplace=True)



In [175]:
def remove_URL(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_emoji(text):
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def remove_punct(text):
    return text.translate(str.maketrans('', '', string.punctuation))



def clean_text(df, text_field):
    df[text_field] = df[text_field].apply(remove_URL)
    df[text_field] = df[text_field].apply(remove_emoji)
    df[text_field] = df[text_field].apply(remove_punct)

    return df

# Now let's clean the 'text' column
train = clean_text(train, 'text')
test = clean_text(test, 'text')



In [176]:
train

,id,keyword,location,text,target
0,1,,,Our Deeds are the Reason of this earthquake May ALLAH Forgive us all,1
1,4,,,Forest fire near La Ronge Sask Canada,1
2,5,,,All residents asked to shelter in place are being notified by officers No other evacuation or shelter in place orders are expected,1
3,6,,,13000 people receive wildfires evacuation orders in California,1
4,7,,,Just got sent this photo from Ruby Alaska as smoke from wildfires pours into a school,1
...,...,...,...,...,...
7608,10869,,,Two giant cranes holding a bridge collapse into nearby homes,1
7609,10870,,,ariaahrary TheTawniest The out of control wild fires in California even in the Northern part of the state Very troubling,1
7610,10871,,,M194 0104 UTC5km S of Volcano Hawaii,1
7611,10872,,,Police investigating after an ebike collided with a car in Little Portugal Ebike rider suffered serious nonlife threatening injuries,1


In [177]:
test

,id,keyword,location,text
0,0,,,Just happened a terrible car crash
1,2,,,Heard about earthquake is different cities stay safe everyone
2,3,,,there is a forest fire at spot pond geese are fleeing across the street I cannot save them all
3,9,,,Apocalypse lighting Spokane wildfires
4,11,,,Typhoon Soudelor kills 28 in China and Taiwan
...,...,...,...,...
3258,10861,,,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTENERS XrWn
3259,10865,,,Storm in RI worse than last hurricane My cityamp3others hardest hit My yard looks like it was bombed Around 20000K still without power
3260,10868,,,Green Line derailment in Chicago
3261,10874,,,MEG issues Hazardous Weather Outlook HWO


In [178]:

# Define features and target
X_train = train.drop('target', axis=1)
y_train = train['target']
X_test = test
# Define the vectorizers
vectorizers = ColumnTransformer(
    transformers=[
        ('keyword_vec', TfidfVectorizer(stop_words='english'), 'keyword'),
        ('text_vec', TfidfVectorizer(stop_words='english'), 'text')
    ]
)

# Apply the vectorizers to the training and test data
X_train_vect = vectorizers.fit_transform(X_train)
X_test_vect = vectorizers.transform(X_test)

# Define the SVM classifier
svm_clf = SVC(kernel='linear', probability=True)

# Fit the classifier on the vectorized training data
svm_clf.fit(X_train_vect, y_train)


SVC(kernel='linear', probability=True)

In [21]:
#get score of the model svm_clf
print(svm_clf.score(X_train_vect, y_train))


0.9295941153290425


In [179]:
predictions = svm_clf.predict(X_test_vect)
predictions
submission = pd.DataFrame({'id': test['id'], 'target': predictions})
submission.to_csv('submission.csv', index=False)


In [18]:
#    # Define the Random Forest classifier
#
#
#    # Define the stratified k-fold cross-validation
#    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#
#    # Perform cross-validation
#    cv_scores = []
#    for train_index, val_index in skf.split(X_train_vect, y_train):
#        X_train_fold, X_val_fold = X_train_vect[train_index], X_train_vect[val_index]
#        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]
#        
#        rf_clf.fit(X_train_fold, y_train_fold)
#        val_predictions = rf_clf.predict(X_val_fold)
#        
#        accuracy = accuracy_score(y_val_fold, val_predictions)
#        f1 = f1_score(y_val_fold, val_predictions, average='weighted')
#        cv_scores.append({'accuracy': accuracy, 'f1': f1})
#
#    # Print the cross-validation scores
#    print("Cross-validation scores:", cv_scores)
#
#    # Fit the Random Forest classifier on the entire training data
#    rf_clf.fit(X_train_vect, y_train)
#    rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
#    # Predict on the test data
#    test_predictions = rf_clf.predict(X_test_vect)
#    submission = pd.DataFrame({'id': test['id'], 'target': test_predictions})
#    submission.to_csv('submission.csv', index=False)

Cross-validation scores: [{'accuracy': 0.7846355876559422, 'f1': 0.7781821088504519}, {'accuracy': 0.7820091923834537, 'f1': 0.7760064274810106}, {'accuracy': 0.7734734077478661, 'f1': 0.7679653114711701}, {'accuracy': 0.7871222076215506, 'f1': 0.7802544327530925}, {'accuracy': 0.7884362680683311, 'f1': 0.7830435704558273}]


In [44]:
#def clean(tweet, http = True, punc = True, lem = True, stop_w = True):
#    
#    if http is True:
#        tweet = re.sub("https?:\/\/t.co\/[A-Za-z0-9]*", '', tweet)
#
#    # stop words
#    # in here I changed the placement of lower for those of you who want to use
#    # Cased BERT later on.
#    if stop_w == 'nltk':
#        tweet = [word for word in word_tokenize(tweet) if not word.lower() in nltk_st]
#        tweet = ' '.join(tweet)
#
#    elif stop_w == 'spacy':
#        tweet = [word for word in word_tokenize(tweet) if not word.lower() in spacy_st]
#        tweet = ' '.join(tweet)
#
#    # lemmitizing
#    if lem == True:
#        lemmatized = [word.lemma_ for word in sp(tweet)]
#        tweet = ' '.join(lemmatized)
#
#    # punctuation removal
#    if punc is True:
#        tweet = tweet.translate(str.maketrans('', '', string.punctuation))
#        
#    # removing extra space
#    tweet = re.sub("\s+", ' ', tweet)
#    
#    return tweet

<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:27: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\/'
<>:27: SyntaxWarning: invalid escape sequence '\s'
C:\Users\eugen\AppData\Local\Temp\ipykernel_11812\1775499501.py:4: SyntaxWarning: invalid escape sequence '\/'
  tweet = re.sub("https?:\/\/t.co\/[A-Za-z0-9]*", '', tweet)
C:\Users\eugen\AppData\Local\Temp\ipykernel_11812\1775499501.py:27: SyntaxWarning: invalid escape sequence '\s'
  tweet = re.sub("\s+", ' ', tweet)


In [45]:
#df_concat['cleaned_text'] = df_concat.text.apply(lambda x: clean(x, lem = False, stop_w = 'nltk', http = True, punc = True))

In [54]:
#cleaned_train = df_concat[:train.shape[0]]
#cleaned_test = df_concat[train.shape[0]:]
#
##drop the text column
#cleaned_train.drop(columns = ['text'], inplace = True)
#cleaned_test.drop(columns = ['text'], inplace = True)
#cleaned_test.drop(columns = ['target'], inplace = True)


C:\Users\eugen\AppData\Local\Temp\ipykernel_11812\788824734.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_train.drop(columns = ['text'], inplace = True)
C:\Users\eugen\AppData\Local\Temp\ipykernel_11812\788824734.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_test.drop(columns = ['text'], inplace = True)
C:\Users\eugen\AppData\Local\Temp\ipykernel_11812\788824734.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-c

In [166]:
#splitting the data



# Define features and target
X_train = cleaned_train.drop('target', axis=1)
y_train = cleaned_train['target']
X_test = cleaned_test

# Define the vectorizers
vectorizers = ColumnTransformer(
    transformers=[
        ('keyword_vec', TfidfVectorizer(stop_words='english'), 'keyword'),
        ('text_vec', TfidfVectorizer(stop_words='english'), 'cleaned_text')
    ]
)

# Apply the vectorizers to the training and test data
X_train_vect = vectorizers.fit_transform(X_train)
X_test_vect = vectorizers.transform(X_test)

# Define the SVM classifier
svm_clf = SVC(kernel='linear', probability=True)

# Fit the classifier on the vectorized training data
svm_clf.fit(X_train_vect, y_train)


SVC(kernel='linear', probability=True)

In [167]:

print(svm_clf.score(X_train_vect, y_train))

0.9269670300801262


In [168]:
test=pd.read_csv('test.csv')
#get score of the model svm_clf
predictions = svm_clf.predict(X_test_vect)
predictions
submission = pd.DataFrame({'id': test['id'], 'target': predictions})
submission.to_csv('submission.csv', index=False)


In [169]:
predictions

array([1., 1., 1., ..., 1., 1., 0.])